# $(SASA) Models - Kmeans$

In [1]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pickle
import numpy as np
import pandas as pd

from pmbrl.model2 import Base_Line_Simple_Model
# from pmbrl.model2 import Model, Regularized_Reference_Loss
from pmbrl.data import Experiment_Data, get_data_expanded

In [2]:
nome_do_arquivo = 'kmodels.pkl'

with open(nome_do_arquivo, 'rb') as arquivo:
    exp = pickle.load(arquivo)
    data = exp['data']
    models = exp['model']

del exp
del arquivo

In [3]:
data = Experiment_Data()

data.load(path='../testing_data.csv')

expansions = {
    's': ['s0', 's1', 's2', 's3'],
    's_': ['s_0', 's_1', 's_2', 's_3'],
    's__': ['s__0', 's__1', 's__2', 's__3']
}

df = get_data_expanded(data.build_training_dataset(), expansions)
# df = df.loc[df['episode']<15].copy()
df.head()

,step,episode,p,s,a,r,s_,a_,r_,s__,...,s2,s3,s_0,s_1,s_2,s_3,s__0,s__1,s__2,s__3
685,6,34,"(0.1912497154781393, 0.2075668863455732)","(-0.049, 0.04, -0.013, -0.039)",1,1.0,"(-0.048, 0.256, -0.014, -0.805)",1.0,1.0,"(-0.043, 0.473, -0.03, -1.572)",...,-0.013,-0.039,-0.048,0.256,-0.014,-0.805,-0.043,0.473,-0.030,-1.572
1727,15,87,"(0.2009601517301062, 0.9751414354812274)","(-0.109, -0.539, 0.066, 0.478)",0,1.0,"(-0.12, -0.729, 0.075, 0.651)",1.0,1.0,"(-0.134, -0.54, 0.088, 0.502)",...,0.066,0.478,-0.120,-0.729,0.075,0.651,-0.134,-0.540,0.088,0.502
1859,11,92,"(0.2225177737903001, 0.4165106988169785)","(0.061, 0.167, -0.146, -0.63)",0,1.0,"(0.064, -0.029, -0.159, -0.31)",0.0,1.0,"(0.064, -0.225, -0.165, 0.004)",...,-0.146,-0.630,0.064,-0.029,-0.159,-0.310,0.064,-0.225,-0.165,0.004
1153,6,54,"(0.1941140178226126, 0.1821930662665186)","(-0.058, -0.439, 0.072, 1.709)",1,1.0,"(-0.067, -0.221, 0.106, 0.915)",0.0,1.0,"(-0.071, -0.446, 0.124, 1.869)",...,0.072,1.709,-0.067,-0.221,0.106,0.915,-0.071,-0.446,0.124,1.869
217,6,14,"(0.4344834695485677, 0.3283152435126403)","(-0.006, -0.439, 0.137, 1.408)",0,1.0,"(-0.015, -0.651, 0.165, 2.073)",0.0,1.0,"(-0.028, -0.862, 0.206, 2.75)",...,0.137,1.408,-0.015,-0.651,0.165,2.073,-0.028,-0.862,0.206,2.750


# Predict 

In [4]:
def evaluate(models, df):
    def predict(model):
        prediction_dataset = df.copy()
        prediction_dataset[model.grouped_targets_lables] = prediction_dataset.apply(lambda row: data._predict_from_row(row, model), axis=1, result_type='expand')
        return prediction_dataset

    predictions = [predict(m) for m in models]

    prediction_dataset = df.copy()
    for i, pred in enumerate(predictions):
        prediction_dataset[f'estimated_s_model_{i}'] = pred['estimated_s']
        
        results = data.get_evaluation_metrics(pred, p=False)
        prediction_dataset[f'rse_model_{i}'] = results['rse']
        prediction_dataset[f'rse_normalized_model_{i}'] = results['rse_normalized']

        prediction_dataset[f'rse_s0_model_{i}'] = results['rse_s0']
        prediction_dataset[f'rse_s1_model_{i}'] = results['rse_s1']
        prediction_dataset[f'rse_s2_model_{i}'] = results['rse_s2']
        prediction_dataset[f'rse_s3_model_{i}'] = results['rse_s3']

        prediction_dataset[f'rse_s0_normalized_model_{i}'] = results['rse_s0_normalized']
        prediction_dataset[f'rse_s1_normalized_model_{i}'] = results['rse_s1_normalized']
        prediction_dataset[f'rse_s2_normalized_model_{i}'] = results['rse_s2_normalized']
        prediction_dataset[f'rse_s3_normalized_model_{i}'] = results['rse_s3_normalized']


    return prediction_dataset

In [5]:
def reagroup(prediction_dataset):
    # agg_results = prediction_dataset[['episode'] + [
    #     f'rse_model_{i}' for i,_ in enumerate(models)
    # ]].groupby('episode').mean().reset_index()
    agg_results = prediction_dataset[['episode', 'step'] + [
        f'rse_model_{i}' for i,_ in enumerate(models)
    ]].copy()
    
    agg_results['best_model'] = agg_results.apply(lambda row: np.argmin(row[2:].values), axis=1)
    print(agg_results['best_model'].value_counts())

    prediction_dataset['best_model'] = prediction_dataset.apply(
        lambda row: agg_results.loc[(agg_results['episode']==row['episode']) & (agg_results['step']==row['step'])].best_model.values[0],
        axis=1
    )
    prediction_dataset['best_rse'] = prediction_dataset.apply(lambda row: row[f'rse_model_{row.best_model}'],axis=1)

    return prediction_dataset

In [6]:
def predicts(models, df):
    pre_df = df.copy()
    pre_df[['s_0', 's_1', 's_2', 's_3', 'a_', 's__0', 's__1', 's__2', 's__3']] = pre_df[['s0', 's1', 's2', 's3', 'a', 's_0', 's_1', 's_2', 's_3']]

    prediction_dataset = evaluate(models, pre_df)
    prediction_dataset = reagroup(prediction_dataset)
    final_predictions = evaluate(models, df)
    final_predictions['group'] = prediction_dataset['best_model']

    cols = [
        'estimated_s', 'rse', 'rse_normalized',
        'rse_s0', 'rse_s1', 'rse_s2', 'rse_s3', 
        'rse_s0_normalized', 'rse_s1_normalized',
        'rse_s2_normalized', 'rse_s3_normalized'
    ]

    for c in cols:
        final_predictions[c] = final_predictions.apply(lambda x: x[f'{c}_model_{x.group}'], axis=1)

    return final_predictions[cols]

In [7]:
prediction_dataset = predicts(models, df)
prediction_dataset.head()

best_model
3    1057
0     349
1     325
4     235
2     161
Name: count, dtype: int64


,estimated_s,rse,rse_normalized,rse_s0,rse_s1,rse_s2,rse_s3,rse_s0_normalized,rse_s1_normalized,rse_s2_normalized,rse_s3_normalized
685,"(-0.034, 0.462, -0.027, -1.275)",0.320,0.513169,0.009,0.011,0.003,0.297,0.555438,0.528761,0.508393,0.460084
1727,"(-0.133, -0.538, 0.088, 0.487)",0.018,0.502673,0.001,0.002,0.000,0.015,0.546990,0.526549,0.501199,0.435954
1859,"(0.066, -0.237, -0.159, 0.291)",0.307,0.512967,0.002,0.012,0.006,0.287,0.548046,0.529007,0.515588,0.459228
1153,"(-0.07, -0.413, 0.125, 1.203)",0.701,0.519104,0.001,0.033,0.001,0.666,0.546990,0.534169,0.503597,0.491657
217,"(-0.007, -0.877, 0.211, 3.052)",0.343,0.517889,0.021,0.015,0.005,0.302,0.568110,0.529744,0.513189,0.460512


In [8]:
prediction_dataset.describe()

,rse,rse_normalized,rse_s0,rse_s1,rse_s2,rse_s3,rse_s0_normalized,rse_s1_normalized,rse_s2_normalized,rse_s3_normalized
count,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000,2127.000000
mean,0.251553,0.511088,0.004172,0.012431,0.003860,0.231091,0.550340,0.529113,0.510455,0.454444
std,0.470486,0.013627,0.007960,0.022851,0.007032,0.445076,0.008405,0.005617,0.016864,0.038083
min,0.001000,0.502008,0.000000,0.000000,0.000000,0.000000,0.545935,0.526057,0.501199,0.434671
25%,0.061000,0.504276,0.001000,0.003000,0.000000,0.054000,0.546990,0.526794,0.501199,0.439292
50%,0.147000,0.507055,0.001000,0.007000,0.001000,0.134000,0.546990,0.527778,0.503597,0.446137
75%,0.277000,0.513077,0.004000,0.013000,0.004000,0.252000,0.550158,0.529253,0.510791,0.456233
max,7.246000,0.694912,0.110000,0.339000,0.082000,6.837000,0.662091,0.609390,0.697842,1.019680
